# Sentence-BERT (SBERT) for Resume–Job Matching

1) Loading the datasets.

In [ ]:
from datasets import load_dataset

# Job descriptions dataset
ds = load_dataset("lang-uk/recruitment-dataset-job-descriptions-english")
df_jobs = ds["train"].to_pandas()
df_jobs.to_csv("job-descriptions.csv", index=False)

# For the resume dataset
# Place CareerCorpus.xlsx in the same working directory before running the notebook

Packages and imports needed for Sentence-BERT (SBERT)

In [ ]:
!pip install sentence-transformers
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

Reading the datasets

In [ ]:
df_resumes = pd.read_excel("CareerCorpus.xlsx")
df_jobs = pd.read_csv("job-descriptions.csv")

Inspecting the datasets

In [ ]:
print(df_resumes.head())
print(df_resumes.columns)

print(df_jobs.head())
print(df_jobs.columns)

         ID   Domain                                          Education  \
0  74552449  Banking  Master of Science in International Trade, Univ...   
1  79041971  Banking  High School Diploma, Federal Way Senior High S...   
2  77156708  Banking  Master of Management, Business Management — Co...   
3  24580361  Banking  B.S. in Operations Management, University of D...   
4  34953092  Banking  M.S. Computer Engineering, University of Misso...   

                             Skills and Achievements  \
0  Skilled in cash handling, loan operations, fin...   
1  Strong leadership, team management, and client...   
2  NMLS #1796859; business development; project m...   
3  Microsoft Office (Excel/PowerPoint/Word/Access...   
4  C/C++, Python, MATLAB, SQL, R, LUA, VBA; ML (s...   

                                          Experience      Job_type  \
0  Professional (11/2016–Present)—opened and mana...   Entry-level   
1  Banking Professional (Aug 2013–Present)—overse...     Mid-level   
2 

Resume dataset relevant columns:
*   *Education*
*   *Skills and Achievements*
*   *Experience*
These together form the CV content.

Jobs dataset relevant columns:
*   *Position*
*   *Long Description*

Define the text for each dataset and combine them

In [ ]:
# For the resume dataset
resumes = df_resumes.copy()

resumes["text"] = (
    resumes["Education"].fillna('') + " " +
    resumes["Skills and Achievements"].fillna('') + " " +
    resumes["Experience"].fillna('')
)

resumes = resumes[["text", "Domain", "Job_type"]]

# For the jobs dataset
jobs = df_jobs.copy()

jobs["text"] = (
    jobs["Position"].fillna('') + " " +
    jobs["Long Description"].fillna('')
)

jobs = jobs[["text", "Primary Keyword"]]

# Removing empty rows
resumes = resumes.dropna(subset=["text"]).copy()
jobs = jobs.dropna(subset=["text"]).copy()

Minimal cleaning

In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

resumes["cleaned"] = resumes["text"].apply(clean_text)
jobs["cleaned"] = jobs["text"].apply(clean_text)

Loading the model

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding resumes and job descriptions turning them into embedding vectors

In [ ]:
# resume encoding
resume_embeddings = model.encode(
    resumes["cleaned"].tolist(),
    show_progress_bar=True
)

#job description dataset is way to big and i figured that i dont need to encode the
#entire dataset for this testing so im only going encode the ones im going to use in testing sections

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

**Testing** **1**

In [ ]:
job_index = 0

#Encoding job description
job_embedding = model.encode([jobs.iloc[job_index]["cleaned"]])
scores = cosine_similarity(job_embedding, resume_embeddings)[0]

print("JOB:")
print(jobs.iloc[job_index]["text"][:500])


resumes["score"] = scores
ranked = resumes.sort_values(by="score", ascending=False)
ranked["preview"] = ranked["text"].str[:300]

print("\nTOP MATCHES:")
print(ranked[["preview", "score"]].head(10))

JOB:
10 + Blockchain Nodes / Masternodes to set up *Requirements*

We're looking for a long term collaboration with someone that has an experience in crypto, masternodes, nodes, validators etc. We need to set up:

Kyber Network
Nebulas
SecretNetwork
Tron
Aion
DeFiChain
EOS
TomoChain
Elrond
IRISnet
IoTeX
Terra
ChainX
Thorchain

Succesful candidates will have an opportunity to get more jobs and long term collaboration.

TOP MATCHES:
                                               preview     score
258  B.Sc. in Computer Science & Engineering, Bangl...  0.424083
297  M.Sc. in CSE (CGPA 3.97/4.0) & B.Sc. in CSE (C...  0.416196
6    BS, Electrical & Computer Engineering—Univ. of...  0.399551
216  Associate in Networking Tech, Bucks County CC ...  0.396569
288  Ph.D. in Computer Science, Univ. of Kentucky (...  0.384991
283  M.S. in Computer Science, Univ. of Nebraska at...  0.380185
34   B.E., Electronics & Communication, Easwari Eng...  0.353651
282  B.Sc. in Computer Science & Engineering,

**Testing** **2**

In [ ]:
job_index = 6

#Encoding the job description
job_embedding = model.encode([jobs.iloc[job_index]["cleaned"]])
scores = cosine_similarity(job_embedding, resume_embeddings)[0]

print("JOB:")
print(jobs.iloc[job_index]["text"][:500])


resumes["score"] = scores
ranked = resumes.sort_values(by="score", ascending=False)
ranked["preview"] = ranked["text"].str[:300]

print("\nTOP MATCHES:")
print(ranked[["preview", "score"]].head(10))

JOB:
1806/03 Sales Manager **Requirements:**
•⠀Experience in IT sales 1+ year
•⠀Generic understanding of IT technologies and Software Development process
•⠀Knowledge of sales techniques, sales markets
•⠀Good communication and negotiation skills
•⠀Fluent written and good spoken English

**We offer:**
•⠀Friendly teams, experienced colleagues, and perfect work equipment
•⠀Opportunities for career growth and raising professional skills
•⠀Competitive compensation and regular results-based salary

TOP MATCHES:
                                               preview     score
19   MBA, Human Resource Management – University of...  0.558477
78   Business/Marketing and Business Administration...  0.544306
233  B.A. International Business, Salem College (20...  0.530702
144  B.S., Business Administration—Accounting, Univ...  0.518146
34   B.E., Electronics & Communication, Easwari Eng...  0.512097
234  B.S. Business Administration, Eastern Nazarene...  0.507791
220  Certificate in Marketing, Temp

**Testing** **3**

In [26]:
job_index = 82

#Encoding job description
job_embedding = model.encode([jobs.iloc[job_index]["cleaned"]])
scores = cosine_similarity(job_embedding, resume_embeddings)[0]

print("JOB:")
print(jobs.iloc[job_index]["text"][:500])


resumes["score"] = scores
ranked = resumes.sort_values(by="score", ascending=False)
ranked["preview"] = ranked["text"].str[:300]

print("\nTOP MATCHES:")
print(ranked[["preview", "score"]].head(10))

JOB:
2D-, 3D-Animator position Onlyplay is a fast-growing gaming product development company. Our team develops high-quality games to launch them on popular platforms.

We are inspired by new ideas and bring trends to the gaming world. For the coming years, we face serious goals and many interesting tasks.

With the aim of reaching new horizons in the gaming industry, we are announcing a contest for the position 2D-, 3D-Animator, who will join us in Kyiv on a full-time basis. We are a product com

TOP MATCHES:
                                               preview     score
264  PhD Student in Mechanical & Robotics Engineeri...  0.446738
257  B.Sc. in Computer Science & Engineering (2020–...  0.383336
274  M.Sc. in Computer Science, Missouri Univ. of S...  0.314396
290  B.Eng. in Mechanical Engineering, Manipal Inte...  0.313393
256  M.Sc. in Computer Science, University of Manit...  0.309975
261  B.Sc. in Software Engineering, Shahjalal Unive...  0.309164
49   MBA, UCLA Anderson (2002

**Testing** **4**

In [27]:
job_index = 24

#Encoding job description
job_embedding = model.encode([jobs.iloc[job_index]["cleaned"]])
scores = cosine_similarity(job_embedding, resume_embeddings)[0]

print("JOB:")
print(jobs.iloc[job_index]["text"][:500])


resumes["score"] = scores
ranked = resumes.sort_values(by="score", ascending=False)
ranked["preview"] = ranked["text"].str[:300]

print("\nTOP MATCHES:")
print(ranked[["preview", "score"]].head(10))

JOB:
1.Middle/Senior Embedded C/C++ The CHI team is looking for a Middle/Senior Embedded C/C++

About the client:
The leading global supplier of embedded and connected software for the automotive industry.
Client`s company develops the award-winning iGO Navigation Engine, which is flexible, designed for automotive use, and assists in prototyping and developing proofs of concept, whether it’s navigation, connectivity, HMI, or the complete in-car experience.
For deep integration, he configures his t

TOP MATCHES:
                                               preview     score
34   B.E., Electronics & Communication, Easwari Eng...  0.443923
251  B.Sc. in CSE, Chittagong University of Enginee...  0.438163
258  B.Sc. in Computer Science & Engineering, Bangl...  0.430489
257  B.Sc. in Computer Science & Engineering (2020–...  0.425157
49   MBA, UCLA Anderson (2002); BE, Thapar Univ. (1...  0.415879
274  M.Sc. in Computer Science, Missouri Univ. of S...  0.409821
41   High School Diploma, Ju

**Testing** **5** (with custom job description)

In [ ]:
custom_job = """
Looking for a data analyst with Python, SQL, data visualization,
machine learning, and statistics experience.
"""
#Encode
custom_clean = clean_text(custom_job)
custom_embedding = model.encode([custom_clean])
scores = cosine_similarity(custom_embedding, resume_embeddings)[0]

#And rank
resumes["score"] = scores
ranked = resumes.sort_values(by="score", ascending=False)
ranked["preview"] = ranked["text"].str[:300]

print("CUSTOM JOB:")
print(custom_job)

print("\nTOP MATCHES:")
print(ranked[["preview", "score"]].head(10))

CUSTOM JOB:

Looking for a data analyst with Python, SQL, data visualization,
machine learning, and statistics experience.


TOP MATCHES:
                                               preview     score
4    M.S. Computer Engineering, University of Misso...  0.439493
48   MBA, St. Mary’s College of California (2012). ...  0.377864
261  B.Sc. in Software Engineering, Shahjalal Unive...  0.346563
295  B.Sc. in Computer Science & Engineering, East ...  0.339242
254  B.Sc. in CSE, East West University (2017–2022)...  0.331298
249  B.S. in Information Systems, Northeastern Univ...  0.325561
291  M.Sc. in EEE (2025–26) & B.Sc. in EEE (2019–24...  0.320012
275  Ph.D. in Computer Science, Univ. of Texas at E...  0.319028
19   MBA, Human Resource Management – University of...  0.318675
256  M.Sc. in Computer Science, University of Manit...  0.311775


Saving all the rankings

In [28]:
k = 10
all_results = []

# dataset jobs
tested_jobs = [0, 6, 82, 24]

for job_index in tested_jobs:
    job_embedding = model.encode([jobs.iloc[job_index]["cleaned"]])
    scores = cosine_similarity(job_embedding, resume_embeddings)[0]
    top_k_idx = np.argsort(scores)[-k:][::-1]

    for rank, idx in enumerate(top_k_idx, start=1):
        all_results.append({
            "job_id": f"dataset_{job_index}",
            "job_type": "dataset",
            "job_text": jobs.iloc[job_index]["text"][:300],
            "rank": rank,
            "resume_index": idx,
            "score": scores[idx],
            "resume_domain": resumes.iloc[idx]["Domain"],
            "resume_preview": resumes.iloc[idx]["text"][:300]
        })

# custom job
custom_clean = clean_text(custom_job)
custom_embedding = model.encode([custom_clean])
scores = cosine_similarity(custom_embedding, resume_embeddings)[0]
top_k_idx = np.argsort(scores)[-k:][::-1]

for rank, idx in enumerate(top_k_idx, start=1):
    all_results.append({
        "job_id": "custom_0",
        "job_type": "custom",
        "job_text": custom_job[:300],
        "rank": rank,
        "resume_index": idx,
        "score": scores[idx],
        "resume_domain": resumes.iloc[idx]["Domain"],
        "resume_preview": resumes.iloc[idx]["text"][:300]
    })

final_df = pd.DataFrame(all_results)
final_df.to_csv("SBERT_Results.csv", index=False)
final_df.head(20)

,job_id,job_type,job_text,rank,resume_index,score,resume_domain,resume_preview
0,dataset_0,dataset,10 + Blockchain Nodes / Masternodes to set up ...,1,258,0.424083,Research Assistant,"B.Sc. in Computer Science & Engineering, Bangl..."
1,dataset_0,dataset,10 + Blockchain Nodes / Masternodes to set up ...,2,297,0.416196,Research Assistant,M.Sc. in CSE (CGPA 3.97/4.0) & B.Sc. in CSE (C...
2,dataset_0,dataset,10 + Blockchain Nodes / Masternodes to set up ...,3,6,0.399551,Banking,"BS, Electrical & Computer Engineering—Univ. of..."
3,dataset_0,dataset,10 + Blockchain Nodes / Masternodes to set up ...,4,216,0.396569,Apparel,"Associate in Networking Tech, Bucks County CC ..."
4,dataset_0,dataset,10 + Blockchain Nodes / Masternodes to set up ...,5,288,0.384991,Research Assistant,"Ph.D. in Computer Science, Univ. of Kentucky (..."
5,dataset_0,dataset,10 + Blockchain Nodes / Masternodes to set up ...,6,283,0.380185,Research Assistant,"M.S. in Computer Science, Univ. of Nebraska at..."
6,dataset_0,dataset,10 + Blockchain Nodes / Masternodes to set up ...,7,34,0.353651,Banking,"B.E., Electronics & Communication, Easwari Eng..."
7,dataset_0,dataset,10 + Blockchain Nodes / Masternodes to set up ...,8,282,0.348593,Research Assistant,"B.Sc. in Computer Science & Engineering, BRAC ..."
8,dataset_0,dataset,10 + Blockchain Nodes / Masternodes to set up ...,9,5,0.340076,Banking,"High School Diploma (Math), University of Cali..."
9,dataset_0,dataset,10 + Blockchain Nodes / Masternodes to set up ...,10,49,0.323931,Banking,"MBA, UCLA Anderson (2002); BE, Thapar Univ. (1..."
